<a href="https://colab.research.google.com/github/Gatwaza/Capstone-Project/blob/main/Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import argparse
import os
import tarfile
import pickle
import sys
from pathlib import Path

# only needed for download, not for inspection
try:
    from huggingface_hub import hf_hub_download, login
    HF_AVAILABLE = True
except ImportError:
    HF_AVAILABLE = False
    print("huggingface_hub not installed. Run: pip install huggingface_hub tqdm")

try:
    from tqdm import tqdm
    TQDM_AVAILABLE = True
except ImportError:
    TQDM_AVAILABLE = False


REPO_ID = "ShunliWang/CPR-Coach"
REPO_TYPE = "dataset"

# These two files are everything a Algo-on-landmarks pipeline needs.
ESSENTIAL_FILES = [
    ("ann.tar.gz",       "128 KB",  "Annotation lists, action labels, train/test splits"),
    ("Keypoints.tar.gz", "2.5 GB",  "Pre-extracted AlphaPose skeleton keypoints (pkl)"),
]

# If you also want a small slice of raw video for sanity-checking pose estimation,
# will uncomment just in case raw video is needed for double checking.
OPTIONAL_VIDEO_SAMPLE = [
    # ("CPR_Dataset_S0.tar.gz.00", "10 GB", "First video shard — single-person, set 0"),
]


def download_file(filename: str, output_dir: Path, token: str | None) -> Path:
    """Download one file from HuggingFace Hub with progress."""
    print(f"\n→ Downloading {filename} ...")
    local_path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        filename=filename,
        local_dir=str(output_dir),
        token=token,
    )
    size_mb = Path(local_path).stat().st_size / (1024 ** 2)
    print(f"  ✓ Saved to {local_path}  ({size_mb:.1f} MB)")
    return Path(local_path)


def extract_tar(tar_path: Path, output_dir: Path) -> None:
    """Extract a tar.gz archive with progress."""
    print(f"\n→ Extracting {tar_path.name} ...")
    with tarfile.open(tar_path, "r:gz") as tar:
        members = tar.getmembers()
        if TQDM_AVAILABLE:
            for member in tqdm(members, desc="  Extracting", unit="file"):
                tar.extract(member, path=output_dir)
        else:
            tar.extractall(path=output_dir)
            print(f"  ✓ Extracted {len(members)} files")
    print(f"  ✓ Done.")


def inspect_keypoints(output_dir: Path) -> None:
    """
    Load and print a summary of the keypoints pkl files so you can verify
    the data structure before writing your feature extraction code.
    """
    keypoints_dir = output_dir / "Keypoints"
    if not keypoints_dir.exists():
        print("\n⚠  Keypoints directory not found. Run extraction first.")
        return

    pkl_files = list(keypoints_dir.glob("*.pkl"))
    if not pkl_files:
        print("\n⚠  No .pkl files found in Keypoints/")
        return

    print("\n" + "═" * 60)
    print("KEYPOINTS DATA INSPECTION")
    print("═" * 60)

    for pkl_file in sorted(pkl_files):
        print(f"\n {pkl_file.name}")
        with open(pkl_file, "rb") as f:
            data = pickle.load(f)

        # Handle common structures: dict, list, numpy array
        if isinstance(data, dict):
            print(f"   Type  : dict  ({len(data)} keys)")
            sample_keys = list(data.keys())[:5]
            print(f"   Keys  : {sample_keys} ...")
            # Peek at first value
            first_val = data[sample_keys[0]]
            print(f"   Value type  : {type(first_val)}")
            if hasattr(first_val, "shape"):
                print(f"   Value shape : {first_val.shape}")
            elif isinstance(first_val, list):
                print(f"   Value length: {len(first_val)}")

        elif isinstance(data, list):
            print(f"   Type   : list  ({len(data)} items)")
            if data:
                item = data[0]
                print(f"   Item type  : {type(item)}")
                if hasattr(item, "shape"):
                    print(f"   Item shape : {item.shape}")

        else:
            print(f"   Type   : {type(data)}")
            if hasattr(data, "shape"):
                print(f"   Shape  : {data.shape}")

    print("\n" + "═" * 60)
    print("Use these shapes to configure your BiLSTM input layer.")
    print("Expected: (num_frames, num_keypoints, coords) per sample")
    print("═" * 60)


def inspect_annotations(output_dir: Path) -> None:
    """Print a summary of annotation files."""
    ann_dir = output_dir / "ann"
    if not ann_dir.exists():
        print("\n⚠  ann/ directory not found.")
        return

    print("\n" + "═" * 60)
    print("ANNOTATION FILES SUMMARY")
    print("═" * 60)

    for txt_file in sorted(ann_dir.glob("*.txt")):
        with open(txt_file) as f:
            lines = f.readlines()
        print(f"\n {txt_file.name}  ({len(lines)} entries)")
        for line in lines[:3]:
            print(f"   {line.strip()}")
        if len(lines) > 3:
            print(f"   ... ({len(lines) - 3} more)")

    action_list = ann_dir / "ActionList.txt"
    if action_list.exists():
        print(f"\n ActionList.txt (error class definitions):")
        with open(action_list) as f:
            for line in f:
                print(f"   {line.strip()}")


def main():
    parser = argparse.ArgumentParser(
        description="Download CPR-Coach keypoints + annotations (~2.5 GB) for model training"
    )
    parser.add_argument(
        "--output_dir", type=str, default="/content/drive/MyDrive/cpr_coach_data",
        help="Local directory to save downloaded files (default: /content/drive/MyDrive/cpr_coach_data)"
    )
    parser.add_argument(
        "--token", type=str, default=None,
        help="HuggingFace token (optional — dataset is public)"
    )
    parser.add_argument(
        "--skip_download", action="store_true",
        help="Skip download and only inspect already-extracted files"
    )
    parser.add_argument(
        "--inspect_only", action="store_true",
        help="Only print data structure summary (no download or extraction)"
    )
    args = parser.parse_args([]) # Colab notebook environments

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print("╔══════════════════════════════════════════════════════╗")
    print("║   CPR-Coach Smart Downloader — Keypoints Strategy   ║")
    print("╚══════════════════════════════════════════════════════╝")
    print(f"\nOutput directory : {output_dir.resolve()}")
    print(f"\nDownloading ONLY:")
    for fname, size, desc in ESSENTIAL_FILES:
        print(f"  • {fname:30s} {size:8s}  {desc}")
    print(f"\nSkipping         : ~447 GB of raw video (not needed for Modeling)")

    if args.inspect_only:
        inspect_keypoints(output_dir)
        inspect_annotations(output_dir)
        return

    if not HF_AVAILABLE:
        print("\n✗ huggingface_hub required. Install with:")
        print("    pip install huggingface_hub tqdm")
        sys.exit(1)

    if args.token:
        login(token=args.token)

    if not args.skip_download:
        files_to_download = [f[0] for f in ESSENTIAL_FILES] + \
                            [f[0] for f in OPTIONAL_VIDEO_SAMPLE]

        downloaded_paths = []
        for filename in files_to_download:
            try:
                path = download_file(filename, output_dir, args.token)
                downloaded_paths.append(path)
            except Exception as e:
                print(f"\n✗ Failed to download {filename}: {e}")
                sys.exit(1)

        print(f"\n✓ All files downloaded to {output_dir.resolve()}")

        # Extract both archives
        for path in downloaded_paths:
            if path.suffix in (".gz", ".tar"):
                try:
                    extract_tar(path, output_dir)
                except Exception as e:
                    print(f"\n✗ Extraction failed for {path.name}: {e}")
                    sys.exit(1)

    # Always inspect after download + extraction
    inspect_keypoints(output_dir)
    inspect_annotations(output_dir)

    print("\n Ready for Model training!")
    print("   Next step: run your feature extraction pipeline on Keypoints/*.pkl")
    print("   Reference: Section 3.3 of your proposal — 12 engineered features per frame")


if __name__ == "__main__":
    main()

╔══════════════════════════════════════════════════════╗
║   CPR-Coach Smart Downloader — Keypoints Strategy   ║
╚══════════════════════════════════════════════════════╝

Output directory : /content/drive/MyDrive/cpr_coach_data

  • ann.tar.gz                     128 KB    Annotation lists, action labels, train/test splits
  • Keypoints.tar.gz               2.5 GB    Pre-extracted AlphaPose skeleton keypoints (pkl)

Skipping         : ~447 GB of raw video (not needed for BiLSTM)

→ Downloading ann.tar.gz ...


ann.tar.gz:   0%|          | 0.00/35.5k [00:00<?, ?B/s]

  ✓ Saved to /content/drive/MyDrive/cpr_coach_data/ann.tar.gz  (0.0 MB)

→ Downloading Keypoints.tar.gz ...


Keypoints.tar.gz:   0%|          | 0.00/2.60G [00:00<?, ?B/s]

  ✓ Saved to /content/drive/MyDrive/cpr_coach_data/Keypoints.tar.gz  (2477.6 MB)

✓ All files downloaded to /content/drive/MyDrive/cpr_coach_data

→ Extracting ann.tar.gz ...


  Extracting:   0%|          | 0/9 [00:00<?, ?file/s]/tmp/ipykernel_2541/3951547512.py:61: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=output_dir)
  Extracting: 100%|██████████| 9/9 [00:00<00:00, 82.30file/s]

  ✓ Done.

→ Extracting Keypoints.tar.gz ...



  Extracting: 100%|██████████| 5/5 [01:18<00:00, 15.68s/file]


  ✓ Done.

════════════════════════════════════════════════════════════
KEYPOINTS DATA INSPECTION
════════════════════════════════════════════════════════════

📄 all_errors_keypoints.pkl
   Type   : list  (3312 items)
   Item type  : <class 'dict'>

📄 double_errors_keypoints.pkl
   Type   : list  (2360 items)
   Item type  : <class 'dict'>

📄 test_keypoints.pkl
   Type   : list  (1008 items)
   Item type  : <class 'dict'>

📄 train_keypoints.pkl
   Type   : list  (1344 items)
   Item type  : <class 'dict'>

════════════════════════════════════════════════════════════
Use these shapes to configure your BiLSTM input layer.
Expected: (num_frames, num_keypoints, coords) per sample
════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
ANNOTATION FILES SUMMARY
════════════════════════════════════════════════════════════

📄 ActionList.txt  (77 entries)
   Correct
   Overlap Hands
   Clenching Hands
   ... (74 more)

📄 all_err

In [ ]:
import os

download_path = '/content/drive/MyDrive/cpr_coach_data'

print(f"Listing contents of {download_path}:")
if os.path.exists(download_path):
    for item in os.listdir(download_path):
        print(f"- {item}")
else:
    print(f"Directory {download_path} does not exist.")

Listing contents of /content/drive/MyDrive/cpr_coach_data:
- .cache
- ann.tar.gz
- Keypoints.tar.gz
- ann
- Keypoints
